In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib inline

In [ ]:
from typing import Union

In [ ]:
from PIL import Image as PImage
from PIL.Image import Image

In [ ]:
from pathlib import Path

In [ ]:
from collections import OrderedDict

In [ ]:
import numpy as np

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn, Tensor
from torchvision import (transforms, datasets)
# from torchvision import prototype as P

## Image search

In [ ]:
from typing import Union
from pathlib import Path
from tqdm import tqdm

In [ ]:
import cv2

In [ ]:
from torch import no_grad
from torch.jit import ScriptModule
from torchvision.models import (resnet34, resnet50, wide_resnet50_2)

In [ ]:
print(torch.__version__)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
device

In [ ]:
size = 256
imsz = 224
IMG_SUFF = {'.jpg', '.jpeg', '.png'}
path = Path('data')

In [ ]:
class ToPILImage(object):
    """Convert inout image to PIL image"""

    def __init__(self, mode=None):
        super().__init__()
        self.to_pil = transforms.ToPILImage(mode=mode)

    def convert(self, img: Union[np.ndarray, Image]) -> Image:
        """
        Converts image to the PIL format
        Args:
            img: inout image

        Returns:
            converted image
        """
        return img if isinstance(img, Image) else self.to_pil(img)

    def __call__(self, *args, **kwargs) -> Image:
        return self.convert(*args, **kwargs)

    def __repr__(self):
        format_string = self.__class__.__name__ + '('
        if self.to_pil.mode is not None:
            format_string += f'mode={self.to_pil.mode}'
        format_string += ')'
        return format_string


class Img2Vec(object):
    """Model wrapper for image embedding"""

    def __init__(
        self, backbone: Union[nn.Module, ScriptModule], trfm: transforms, device: str = 'cpu', 
        func:callable = None, ptrf:callable=None):
        super().__init__()
        self.device = torch.device(device)
        self.backbone = (backbone.eval() if hasattr(backbone, 'eval') else backbone).to(device)
        self.call_backbone = func if func else self.backbone
        self.trfm = trfm
        self.ptrf = ptrf

    def preprocess(self, *xs: Union[Image, np.ndarray]) -> Tensor:
        """
        Transform data before model
        Args:
            *xs: input data

        Returns:
            processed data for model
        """
        return torch.stack([self.trfm(x) for x in xs]).to(self.device)

    @no_grad()
    def forward(self, *xs: Union[Image, np.ndarray]) -> np.ndarray:
        tns = self.preprocess(*xs)
        rts = self.call_backbone(tns)
        rts = self.ptrf(rts) if self.ptrf else rts
        y = rts.cpu().data.numpy()

        return y

    def __call__(self, *args, **kwargs) -> np.ndarray:
        return self.forward(*args, **kwargs)

In [ ]:
vec_trsfm = transforms.Compose([ToPILImage(mode='RGB'),
                                transforms.Resize(size),
                                transforms.CenterCrop(imsz),
                                transforms.ToTensor(),
                                transforms.Normalize(
                                    mean=[0.485, 0.456, 0.406], 
                                    std=[0.229, 0.224, 0.225])])

#### Prepare data

In [ ]:
search_path = path / 'search'

In [ ]:
dir_paths = [dp for dp in search_path.iterdir() if dp.is_dir()]

In [ ]:
dir_paths

In [ ]:
img_pts = [im_pt for dp in dir_paths for im_pt in dp.iterdir() if im_pt.suffix in IMG_SUFF]

In [ ]:
def read_img(im_pt):
    img = cv2.imread(str(im_pt), cv2.IMREAD_ANYCOLOR)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    
    return img

In [ ]:
def read_pil_img(im_pt):
    img = PImage.open(im_pt)
    
    return img

In [ ]:
imgs = [read_img(ip) for ip in img_pts]

In [ ]:
pil_imgs = [read_pil_img(ip) for ip in img_pts]

#### Compare vectors

In [ ]:
from scipy.spatial.distance import cosine

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
def top_vecs(qi, top_k=5, model=None, comp_vecs=None, normalize:bool=False):
    qv = model(qi)
    qv = qv / np.linalg.norm(qv) if normalize else qv
    qv = qv
    resul_pts = [(cosine(qv, vc / np.linalg.norm(vc) if normalize else vc ), pt) for pt, vc in comp_vecs]
    resul_pts = sorted(resul_pts, key=lambda x: x[0], reverse=False)
    resul_pts = resul_pts[:top_k]
    
    return resul_pts

#### Initialize features extractor

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
from transformers import AutoModel, AutoProcessor
import open_clip

In [ ]:
# Load model and processor
model_id = "jinaai/jina-clip-v2"
model = AutoModel.from_pretrained(model_id, trust_remote_code=True)
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

In [ ]:
st_model = SentenceTransformer("jinaai/jina-clip-v2", trust_remote_code=True)

In [ ]:
img_procc = processor(pil_imgs)

In [ ]:
img_procc

In [ ]:
with tqdm(pil_imgs) as ppl_imgs:
    img_vecs = [pmg, model.encode_image(pmg) for pmg in ppl_imgs]

In [ ]:
query_vec = model.encode_text('Image of river')

In [ ]:
query_vec

In [ ]:
model.

In [ ]:
for img_vec in img_vecs:
    print(cosine(query_vec, img_vec[1]))

In [ ]:
res = top_vecs('A snow', model=model.encode_text, comp_vecs=img_vecs, normalize=True)

In [ ]:
for dist, res_img in res:
    plt.title(f'dist={dist}')
    plt.imshow(res_img)
    plt.show()
    plt.close()